# 02 — ABSA Extraction & Evaluasi Kualitatif
**Tujuan:** Menjalankan ekstraksi aspek + sentimen dan mengevaluasi kualitas prediksi model IndoBERT.

---

In [ ]:
import os
import sys
# Auto-adjust CWD to project root if executed from notebooks directory
if os.getcwd().endswith('notebooks'):
    os.chdir('..')

import pandas as pd
import json
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sys, os
sys.path.insert(0, 'src')

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})
PALETTE = {'positive': '#4CAF50', 'neutral': '#FFC107', 'negative': '#F44336'}

ABSA_PATH = 'data/processed/absa_results.json'
with open(ABSA_PATH, encoding='utf-8') as f:
    absa = json.load(f)

df_absa = pd.DataFrame(absa)
print(f'Total entri ABSA: {len(df_absa)}')
df_absa.head(3)

## 1. Distribusi Sentimen

In [ ]:
sent_counts = df_absa['sentiment_label'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors = [PALETTE.get(l, '#9E9E9E') for l in sent_counts.index]
axes[0].bar(sent_counts.index, sent_counts.values, color=colors, edgecolor='white')
for i, (label, val) in enumerate(sent_counts.items()):
    axes[0].text(i, val + 1, str(val), ha='center', fontsize=11)
axes[0].set_title('Distribusi Sentimen (Count)'); axes[0].set_ylabel('Jumlah Ulasan')

axes[1].pie(sent_counts.values, labels=sent_counts.index, colors=colors,
            autopct='%1.1f%%', startangle=90,
            wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Proporsi Sentimen')

plt.suptitle('Distribusi Sentimen Hasil ABSA IndoBERT', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('results/figures/sentiment_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Confidence Score Model

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(df_absa['confidence'], bins=25, color='#5C6BC0', edgecolor='white')
mean_conf = df_absa['confidence'].mean()
axes[0].axvline(mean_conf, color='#E53935', linestyle='--', linewidth=2,
                label=f'Mean: {mean_conf:.3f}')
axes[0].set_xlabel('Confidence Score'); axes[0].set_ylabel('Frekuensi')
axes[0].set_title('Distribusi Confidence Score'); axes[0].legend()

for sent in ['positive', 'neutral', 'negative']:
    subset = df_absa[df_absa['sentiment_label'] == sent]['confidence']
    if len(subset):
        axes[1].hist(subset, bins=20, alpha=0.6, label=sent, color=PALETTE[sent])
axes[1].set_xlabel('Confidence Score'); axes[1].set_ylabel('Frekuensi')
axes[1].set_title('Confidence per Sentimen'); axes[1].legend()

plt.tight_layout()
plt.savefig('results/figures/confidence_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(df_absa.groupby('sentiment_label')['confidence'].describe().round(3))

## 3. Breakdown Sentimen per Aspek

In [ ]:
ASPECTS = ['harga', 'kualitas', 'pengiriman', 'packing', 'pelayanan']
rows = []
for entry in absa:
    for asp in entry.get('aspects', []):
        rows.append({'aspek': asp, 'sentimen': entry.get('sentiment_label', 'neutral')})

df_asp = pd.DataFrame(rows)
ct = df_asp.groupby(['aspek','sentimen']).size().unstack(fill_value=0)
ct = ct.reindex(columns=['positive','neutral','negative'], fill_value=0)

ax = ct.plot(kind='bar', stacked=True, figsize=(10, 6),
             color=[PALETTE['positive'], PALETTE['neutral'], PALETTE['negative']],
             edgecolor='white', linewidth=0.5)
ax.set_xlabel('Aspek'); ax.set_ylabel('Jumlah Ulasan')
ax.set_title('Distribusi Sentimen per Aspek', fontsize=13, fontweight='bold')
ax.legend(title='Sentimen', labels=['Positif','Netral','Negatif'])
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.savefig('results/figures/aspect_sentiment_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()
print(ct)

## 4. Evaluasi Kualitatif — Sample Review per Sentimen

In [ ]:
N_SAMPLE = 5
for sentiment in ['positive', 'neutral', 'negative']:
    subset = df_absa[df_absa['sentiment_label'] == sentiment]
    print(f'\n{'='*60}')
    print(f'SENTIMEN: {sentiment.upper()} (total: {len(subset)})')
    print('='*60)
    sample = subset.sample(min(N_SAMPLE, len(subset)), random_state=42)
    for _, row in sample.iterrows():
        print(f"  [{row['confidence']:.2f}] {row['text']}")
        print(f"         Aspek: {row['aspects']}")
        print()

## 5. Validasi: Sentimen ABSA vs Rating Bintang

In [ ]:
CLEAN_PATH = 'data/processed/reviews_clean.csv'
df_clean = pd.read_csv(CLEAN_PATH)

if 'rating' in df_clean.columns:
    df_merged = df_absa.merge(
        df_clean[['text', 'rating']], on='text', how='left'
    ).dropna(subset=['rating'])
    df_merged['rating'] = df_merged['rating'].astype(int)

    ct_val = pd.crosstab(df_merged['rating'], df_merged['sentiment_label'])

    fig, ax = plt.subplots(figsize=(9, 5))
    ct_val.plot(kind='bar', ax=ax, stacked=True,
                color=[PALETTE['negative'], PALETTE['neutral'], PALETTE['positive']],
                edgecolor='white')
    ax.set_xlabel('Rating Bintang'); ax.set_ylabel('Jumlah Ulasan')
    ax.set_title('Sentimen ABSA vs Rating Bintang (Validasi)', fontweight='bold')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    plt.tight_layout()
    plt.savefig('results/figures/absa_vs_rating_validation.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(ct_val)
else:
    print('Kolom rating tidak tersedia untuk validasi.')